# Exercise 2: Related and unrelated sources

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Cross-Lingual Transfer · version 0.1.0*

> Compare related and unrelated source training at matched data and compute budgets. Keep target test data out of proxy and hyperparameter selection.

**Setup.** `pip install sklearn-crfsuite scikit-learn sacrebleu torch transformers sentencepiece fasttext-wheel "stanza==1.11.0" requests "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this. Data come from commit-pinned GitHub repositories and are hash-checked: KazNERD and KyrgyzNER (Exercise 1), SIB-200 (Exercises 2 and 3), the X-WMT Kazakh–Kyrgyz test set (Exercise 4) and the UD 2.17 Turkish–German SAGT and Turkish IMST test files (Exercise 5). Sentence encoders and NLLB-200 are read from local copies (`MODELS_DIR`, `HF_HUB_DIR`, `NLLB_PATH`) or the Hub at the revisions pinned in the code; GlotLID from `GLOTLID_PATH`. Run the notebooks from the `exercises/` folder: the first code cell imports the chapter's helpers from `../code/remaining/transfer.py`. Embeddings, translations and model outputs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`). Everything runs on the CPU.

SIB-200 is parallel, so every source language has the same 701 training sentences with the same topic labels. A logistic-regression classifier on frozen sentence vectors (three encoders) is trained on one source, with C chosen on that source's development split, and tested once on Kyrgyz and Tatar. A paired bootstrap over Kyrgyz test items estimates item-sampling uncertainty for the Kazakh–English source comparison.

In [ ]:
import collections, csv, hashlib, io, json, math, os, random, re, statistics, sys
from pathlib import Path
import numpy as np, requests
sys.path.insert(0, str(Path("../code/remaining").resolve()))
from transfer import SCHEMA, bio_spans, conversion_map, restore_span

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_{}.txt"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/data/{}"
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/xwmt/test/"
UD = "https://raw.githubusercontent.com/UniversalDependencies/{}/r2.17/{}"
SHA256 = {
    KAZNERD.format("train"): "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt"): "57a35a92139497819a2792d3e9e00a41b387b029b7851be7a64f4968befaddf5",
    UD.format("UD_Turkish_German-SAGT", "qtd_sagt-ud-test.conllu"): "0bbb06de66a490bfbb93d103a03d0d89e3835baf0af755a57b5fa5f885ae01bc",
    UD.format("UD_Turkish-IMST", "tr_imst-ud-test.conllu"): "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
}
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256(path.read_bytes()).hexdigest(); FETCHED[url] = h
    if url in SHA256 and h != SHA256[url]:
        raise ValueError(f"{url} differs from the pinned file")
    return path

_CTS = {}
def to_cts(lang, text):
    """Common Turkic Script view (TurkicNLP transliterator); Turkish and English stay unchanged."""
    if lang in ("tur", "eng"):
        return text
    if lang not in _CTS:
        from turkicnlp.scripts import Script
        from turkicnlp.scripts.transliterator import Transliterator
        src = Script.LATIN if lang in ("uzb", "aze") else Script.CYRILLIC
        _CTS[lang] = Transliterator(lang, src, Script.COMMON_TURKIC)
    return _CTS[lang].transliterate(text)

def sib(lang):
    """All 1,004 SIB-200 rows of a language as (index_id, split, topic, text), sorted by index_id."""
    rows = []
    for split in ("train", "dev", "test"):
        text = fetch(f"{SIB}{lang}/{split}.tsv").read_text(encoding="utf-8")
        rows += [(int(r["index_id"]), split, r["category"], r["text"]) for r in csv.DictReader(io.StringIO(text), delimiter="\t")]
    return sorted(rows)

MODELS = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
          "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "c2e19453aae46f98a865abfdf173f35f2e119f5a"}
NAMES = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "MiniLM",
         "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "XLM-R para", "nllb": "NLLB enc"}
NLLB_PATH = os.environ.get("NLLB_PATH", str(Path("~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M").expanduser()))
NLLB_CODE = {"kaz_Cyrl": "kaz_Cyrl", "kir_Cyrl": "kir_Cyrl", "tat_Cyrl": "tat_Cyrl", "bak_Cyrl": "bak_Cyrl", "uzn_Latn": "uzn_Latn",
             "tur_Latn": "tur_Latn", "rus_Cyrl": "rus_Cyrl", "eng_Latn": "eng_Latn", "fra_Latn": "fra_Latn", "azj_Latn": "azj_Latn"}
def model_source(name):
    rev = MODELS[name]
    for p in (Path(os.environ.get("HF_HUB_DIR", "~/.cache/huggingface/hub")).expanduser() / f"models--{name.replace('/', '--')}" / "snapshots" / rev,
              Path(os.environ.get("MODELS_DIR", CACHE / "models")) / f"models--{name.replace('/', '--')}--{rev}"):
        if (p / "config.json").exists():
            return str(p), None
    return name, rev
_ENC = {}
def embed(model, texts, lang=None, batch=32):
    """L2-normalised mean-pooled sentence vectors, as in Meaning and Retrieval (same cache)."""
    key = hashlib.sha1(json.dumps([model, lang, texts], ensure_ascii=False).encode()).hexdigest()[:20]
    path = CACHE / "ch13-vectors" / f"{key}.npy"
    if path.exists():
        return np.load(path)
    import torch
    torch.set_num_threads(THREADS)
    if model not in _ENC:
        from transformers import AutoModel, AutoTokenizer
        if model == "nllb":
            from transformers import AutoModelForSeq2SeqLM
            _ENC[model] = (AutoTokenizer.from_pretrained(NLLB_PATH), AutoModelForSeq2SeqLM.from_pretrained(NLLB_PATH).get_encoder().eval(), 512)
        else:
            src, rev = model_source(model)
            _ENC[model] = (AutoTokenizer.from_pretrained(src, revision=rev), AutoModel.from_pretrained(src, revision=rev).eval(), 128)
    tok, enc, limit = _ENC[model]
    if model == "nllb":
        tok.src_lang = NLLB_CODE[lang]
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch):
            b = tok(texts[i:i + batch], padding=True, truncation=True, max_length=limit, return_tensors="pt")
            h = enc(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            out.append(torch.nn.functional.normalize((h * m).sum(1) / m.sum(1).clamp(min=1), dim=-1).numpy())
    vec = np.concatenate(out).astype(np.float32)
    path.parent.mkdir(parents=True, exist_ok=True); np.save(path, vec)
    return vec

def macro_f1(gold, pred, labels):
    from sklearn.metrics import f1_score
    return f1_score(gold, pred, labels=labels, average="macro", zero_division=0)

print("helpers loaded from code/remaining/transfer.py; schema map:", SCHEMA)

In [ ]:
from sklearn.linear_model import LogisticRegression
TARGETS = ["kir_Cyrl", "tat_Cyrl"]
SOURCES = ["kaz_Cyrl", "tur_Latn", "uzn_Latn", "rus_Cyrl", "eng_Latn", "fra_Latn"]
ENCODERS = ["sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "sentence-transformers/paraphrase-xlm-r-multilingual-v1", "nllb"]
ROWS = {l: sib(l) for l in TARGETS + SOURCES}
assert len({tuple(r[0] for r in v) for v in ROWS.values()}) == 1          # the same sentences, in every language
LABELS = sorted({r[2] for r in ROWS["eng_Latn"]})
split = [r[1] for r in ROWS["eng_Latn"]]; y = np.array([LABELS.index(r[2]) for r in ROWS["eng_Latn"]])
idx = {s: np.array([i for i, x in enumerate(split) if x == s]) for s in ("train", "dev", "test")}
print(f"SIB-200: {len(idx['train'])} training, {len(idx['dev'])} development and {len(idx['test'])} test sentences per language; "
      "logistic regression on frozen sentence vectors, C chosen on the source language's development split")
preds = {}
for m in ENCODERS:
    V = {l: embed(m, [r[3] for r in ROWS[l]], l) for l in ROWS}
    res = {}
    for s in SOURCES:
        best = max((0.1, 1.0, 10.0), key=lambda C: macro_f1(y[idx["dev"]], LogisticRegression(C=C, max_iter=2000).fit(
            V[s][idx["train"]], y[idx["train"]]).predict(V[s][idx["dev"]]), list(range(len(LABELS)))))
        clf = LogisticRegression(C=best, max_iter=2000).fit(V[s][idx["train"]], y[idx["train"]])
        res[s] = {t: macro_f1(y[idx["test"]], clf.predict(V[t][idx["test"]]), list(range(len(LABELS)))) for t in TARGETS + [s]}
        preds[m, s] = clf.predict(V["kir_Cyrl"][idx["test"]])
    print(f"\n{NAMES[m]}: test macro-F1 (rows: source of the training labels)")
    print(f"  {'source':<10}" + "".join(f"{t:>10}" for t in TARGETS) + f"{'source':>10}")
    for s in SOURCES:
        print(f"  {s:<10}" + "".join(f"{res[s][t]:10.3f}" for t in TARGETS) + f"{res[s][s]:10.3f}")
# How large is the noise? Paired bootstrap over the 204 Kyrgyz test items: Kazakh-trained minus English-trained classifier.
rng = np.random.default_rng(0); g = y[idx["test"]]
print("\nKyrgyz test, macro-F1 difference kaz-source minus eng-source, 95% paired bootstrap interval (1,000 resamples):")
for m in ENCODERS:
    a, b = preds[m, "kaz_Cyrl"], preds[m, "eng_Latn"]; d = []
    for _ in range(1000):
        i = rng.integers(len(g), size=len(g))
        d.append(macro_f1(g[i], a[i], list(range(len(LABELS)))) - macro_f1(g[i], b[i], list(range(len(LABELS)))))
    print(f"  {NAMES[m]:<11} {macro_f1(g, a, list(range(len(LABELS)))) - macro_f1(g, b, list(range(len(LABELS)))):+.3f}  [{np.percentile(d, 2.5):+.3f}, {np.percentile(d, 97.5):+.3f}]")

## Worked solution

Output of `experiments/ch16_exercises/run_ch16_exercises.py`:

```
SIB-200: 701 training, 99 development and 204 test sentences per language; logistic regression on frozen sentence vectors, C chosen on the source language's development split

MiniLM: test macro-F1 (rows: source of the training labels)
  source      kir_Cyrl  tat_Cyrl    source
  kaz_Cyrl       0.683     0.595     0.700
  tur_Latn       0.538     0.534     0.871
  uzn_Latn       0.591     0.565     0.613
  rus_Cyrl       0.531     0.536     0.872
  eng_Latn       0.568     0.566     0.866
  fra_Latn       0.552     0.560     0.873

XLM-R para: test macro-F1 (rows: source of the training labels)
  source      kir_Cyrl  tat_Cyrl    source
  kaz_Cyrl       0.811     0.633     0.838
  tur_Latn       0.787     0.601     0.859
  uzn_Latn       0.730     0.588     0.670
  rus_Cyrl       0.787     0.610     0.852
  eng_Latn       0.768     0.619     0.882
  fra_Latn       0.780     0.627     0.882

NLLB enc: test macro-F1 (rows: source of the training labels)
  source      kir_Cyrl  tat_Cyrl    source
  kaz_Cyrl       0.863     0.844     0.846
  tur_Latn       0.879     0.857     0.863
  uzn_Latn       0.857     0.862     0.856
  rus_Cyrl       0.797     0.809     0.813
  eng_Latn       0.823     0.815     0.802
  fra_Latn       0.869     0.866     0.881

Kyrgyz test, macro-F1 difference kaz-source minus eng-source, 95% paired bootstrap interval (1,000 resamples):
  MiniLM      +0.116  [+0.053, +0.180]
  XLM-R para  +0.043  [+0.005, +0.084]
  NLLB enc    +0.040  [-0.001, +0.086]
```

**Source-language rankings depend on the encoder.** On Kyrgyz, a Kazakh-trained MiniLM classifier scores macro-F1 0.683, compared with 0.568 for English training. The paired bootstrap difference is +0.116 [0.053, 0.180]. With XLM-R it is +0.043 [0.005, 0.084]; with the NLLB encoder it is +0.040 [−0.001, 0.086]. The XLM-R interval excludes zero; the NLLB interval does not. Neither result establishes equivalence between sources.

Each source uses 701 parallel training items and the same classifier grid, selected on source development data. This controls task content and classifier settings, but not encoder pretraining exposure or representation quality. Paraphrase encoders use a 128-token ceiling and NLLB 512, as in Meaning and Retrieval's exercises. Training examples are matched; compute costs are not measured as equal.

The table does not establish that genealogical proximity caused a gain. Its intervals cover item resampling for the Kazakh–English comparison, not training variation or all pairwise source comparisons.

## Your turn — extension

Add Azerbaijani and Bashkir as sources and Uyghur (Arabic script) as a target, and test whether the best source for each target is stable across the three encoders. Report bootstrap intervals for each comparison you interpret.